# Field validation — `stratification` (DEPTH pipeline)

| | |
|---|---|
| Subset | `stratification` |
| Pipeline | DEPTH |
| Timestep | 2012-11-09 12:00:00 |
| Domain | one 720 × 720 × 51 tile (≈1400 × 1400 km), set in Section 1 |
| Depth levels | `sfc`, `z25m`, `mld`, `mld_mean` |
| Data | computed on the fly from `s3://dbof/LLC4320_RAW/DEPTH/` |
| Plan | `prompts/field_validation_depth.md` |
| Field reference | `docs/Fields.md` |

Rows of the map and PDF figures are **depth levels**, not regions — that
is the one structural difference from the surface notebooks.

This is the **template** depth notebook: the shortest chain in the DEPTH pipeline, so the machinery is visible rather than buried.  MLD is validated HERE and referenced by every other depth notebook — all the `_mld` and `_mld_mean` channels in the project rest on it.

## Section 1 — Setup

Everything configurable is in the next cell: the **region**, the date,
the depth levels, and the zoom size.  Change `REGION` to validate a
different part of the ocean — any key in `dbof.plotting.regions.REGIONS`
that carries a `zoom` anchor.

Default is the Gulf Stream, anchored at 60°W / 37°N — dynamically
active in every field this project computes, and the same point the
surface notebooks zoom into, so surface and depth look at the same
water.


In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"  # the only DEPTH date transferred so far
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0                  # -> a 200 x 200 km zoom box

SUBSET   = "stratification"
PIPELINE = "DEPTH"
RAW_VARS = ["Theta", "Salt"]

# Profiles (Figure 3)
N_PROFILES        = 5       # <= 5; the fixed location colours are not cycled
PROFILE_SEED      = 42      # same 5 columns for every field in the notebook
PROFILE_MAX_DEPTH = 500.0   # depth-axis limit, m; None = full 969 m column
# ------------------------------------------------------------------------

import dask
import numpy as np

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
import dbof.utils.native_gradient as NG
from dbof.preprocessing.vertical_helpers import (
    _interp_w_to_tracer_levels, _vertical_derivative,
)
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile
from dbof.global_dataset_creation.subset_definitions import (
    get_compute_fn, get_subset_definition, expand_channels_with_suffixes,
)

# tile_utils sets the Agg backend when it is imported (it writes QA PNGs
# on headless nodes), so switch back to inline AFTER the dbof imports or
# no figure in this notebook will render.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

# Channel list straight from the pipeline's own definition -- if the
# subset gains a channel, this notebook picks it up without an edit.
defn = get_subset_definition(PIPELINE, SUBSET)
CHANNELS = expand_channels_with_suffixes(
    defn["compute_features_channels"], list(LEVELS),
    defn.get("extra_channels"),
)
print(f"subset   : {SUBSET}")
print(f"channels : {CHANNELS}")

## Section 2 — Load the tile and compute the fields

We do **not** run `generate-global` here.  That would compute the whole
planet in order to look at one place.

Instead this notebook works on **one tile** — the 720 × 720 × 51 block
the `dbof.tiles` workflow already defines: one LLC face, the full water
column, about 1400 × 1400 km, centred on the region's anchor.  A tile is
*exactly one chunk* of the depth store, so loading it costs one S3 GET
per variable (~106 MB per 3D field).  Tiles are 720-aligned and faces
are 6 × 720 wide, so a tile can never straddle two faces.

Then the **production** compute function for this subset runs on it,
and internally applies the four depth strategies.  Same code as
production, one tile's worth of data.

One thing this costs us: the tile's xgcm grid has **no face
connections**, so cells near the boundary have no neighbours and their
horizontal gradients are wrong.  That rim is NaN'd, using the per-field
widths `tiles/field_registry.py` already records (0 for purely vertical
fields, 1 for staggered interpolation, 3 for gradient and Jacobian
chains).

**A tile samples the region, it does not cover it.**  "Gulf Stream"
here means the ~1400 km tile around 60°W / 37°N — not the whole
80–40°W box the surface notebooks use as a row.


In [ ]:
# Anchor -> rect pixel -> the tile that contains it.
S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)

i_rect, j_rect = tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3)
tile = rect_ij_to_tile(i_rect, j_rect)
print(f"region : {REGION} anchored at ({ANCHOR_LON}, {ANCHOR_LAT})")
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}, "
      f"j={tile.j_face_slice}, i={tile.i_face_slice}")

# Load the tile + its grid, then merge and build a LOCAL xgcm grid.
ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(S3, DATE, tile, RAW_VARS)
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)
print(f"extent : lon [{XC.min():.2f}, {XC.max():.2f}], "
      f"lat [{YC.min():.2f}, {YC.max():.2f}], "
      f"land {100 * LAND.mean():.1f}%")

### The finals, and the intermediates the figures need

`get_compute_fn("DEPTH", SUBSET)` is the production entry point — the
same callable `generate-global` dispatches to — so the finals below are
the pipeline's own numbers.

The **intermediates** are a different matter: the global products never
store them, so they are recomputed here from the same `ds_merge` the
finals came from.  That is deliberate — it means each figure's chain
shows the actual steps, not a reconstruction.


In [ ]:
# Production finals for this subset, on the tile.
store = get_compute_fn(PIPELINE, SUBSET)(ds_merge, xgrid, CHANNELS)
print(f"computed : {sorted(store)}")

# The chain's raw inputs and intermediates.  The global products never
# keep these, so they are recomputed here from the same ds_merge the
# finals came from -- the figures then show the actual steps.
mld = CFAD.mixed_layer_depth(ds_merge)
rho = CF.potential_density(ds_merge)

PROFILE_3D = {
    "Theta": ds_merge["Theta"],
    "Salt":  ds_merge["Salt"],
    "rho":   rho,
    # drho/dz is the entire content of N2 -- N2 is just (g/rho0) times
    # this -- so it earns its own column.  _vertical_derivative is the
    # single canonical implementation; buoyancy_frequency_squared calls
    # exactly this, so the column really is the pipeline's own step.
    "drho_dz": _vertical_derivative(rho, ds_merge),
    # N2 itself, kept lazy here so Figure 3 can profile it too.
    "N2": CFAD.buoyancy_frequency_squared(ds_merge),
}

# Everything except N2 (which comes from `store` at the four levels).
live = dfig.compute_levels(
    {k: v for k, v in PROFILE_3D.items() if k != "N2"},
    ds_merge, mld=mld, levels=LEVELS)

In [ ]:
# How wide the invalid rim is for this subset, straight from the tile
# registry (0 here: nothing in this chain takes a horizontal gradient).
EDGE_MARGIN = dfig.edge_margin_for(
    list(defn["compute_features_channels"])
    + list(defn.get("extra_channels") or []))

# NaN that rim, mask land with the surface hFacC (what production does),
# and reshape into the {base: {level: (x, y, arr)}} the figures take.
level_arrays = dfig.pack_tile_levels(
    {**live, **store}, XC, YC, edge_margin=EDGE_MARGIN,
    land_mask=LAND, levels=LEVELS)

In [ ]:
# Five ocean columns, seeded and spread across the tile, reused by every
# field in this notebook so the profile panels are comparable.
POINTS = dfig.pick_profile_points(
    LAND, n=N_PROFILES, edge_margin=max(EDGE_MARGIN, 1),
    seed=PROFILE_SEED)

# Full water column at those five points -- a few hundred numbers per
# field, so this is cheap next to the maps.
PROFILES, DEPTH_M = dfig.sample_profiles(PROFILE_3D, ds_merge, POINTS)

# MLD at each point, to mark on the profiles.
MLD_AT_POINTS = (
    [level_arrays["mixed_layer_depth"]["sfc"][2][j, i] for j, i in POINTS]
    if "mixed_layer_depth" in level_arrays else None)

## Section 3 — Subset: `stratification`

Channels, verbatim from `subset_definitions.DEPTH_SUBSETS`:

| Channel | Kind |
|---|---|
| `N2_sfc`, `N2_z25m`, `N2_mld`, `N2_mld_mean` | base × depth suffixes |
| `mixed_layer_depth` | extra (inherently 2D) |
| `ml_heat_content` | extra (inherently 2D) |

The two extras integrate over the whole water column, so they have no
depth dependence — their figures collapse to two rows (whole tile and
zoom) rather than repeating the same map four times.


## Section 4 — Field & dependency table

| FIELD | UNITS | EQUATION | DEPENDS ON | CODE |
|---|---|---|---|---|
| `rho` | kg m⁻³ | ρ = JMD95(S, Θ, p = 0) | Theta, Salt | `calculate_fields.potential_density` |
| `drho_dz` | kg m⁻⁴ | ∂ρ/∂z, centred difference (one-sided at the ends) | rho, Z | `vertical_helpers._vertical_derivative` |
| `N2_{sfx}` | s⁻² | N² = (g/ρ₀)·∂ρ/∂z | drho_dz | `calculate_fields_at_depth.buoyancy_frequency_squared` |
| `mixed_layer_depth` | m | deepest z with σ₀ − σ₀(10 m) ≤ 0.03 kg m⁻³ | rho, Z | `calculate_fields_at_depth.mixed_layer_depth` |
| `ml_heat_content` | J m⁻² | Q = ∫₀^MLD c_p·ρ₀·Θ dz | Theta, MLD, drF | `calculate_fields_at_depth.mixed_layer_heat_content` |

σ₀ is `rho − 1000`; the constant drops out of both the vertical
derivative and the MLD threshold, so the figures plot `rho` and the two
are interchangeable here.

### The sign of N², and which way z points

This is the one thing in this subset that reliably confuses people, so
it is worth being explicit.

The textbook form is **N² = −(g/ρ₀)·∂ρ/∂z**, which assumes z is
**positive upward** (z < 0 below the surface).  Density then *decreases*
with increasing z, so ∂ρ/∂z < 0 and the leading minus makes N² > 0.

This code uses the opposite convention.  `_get_depth_coord` returns
|Z| — **depth, positive downward**, 0 at the surface and increasing as
you go down.  Density *increases* with depth, so ∂ρ/∂z > 0 and

**N² = +(g/ρ₀)·∂ρ/∂z**

with no leading minus.  Same physics, same numbers; the sign of the
constant flips with the sign convention of z, and the two conventions
cancel out.  `drho_dz` gets its own column above precisely so this is
checkable by eye: in a stable water column that panel should be
**positive**, and the N² panel should have the same sign everywhere.

**Processing operations in play:** depth selection and averaging
(`depth_strategies`: k = 0 / nearest-k to 25 m / nearest-k to MLD /
thickness-weighted mean over the mixed layer), the vertical derivative
on the tracer grid (float64 internally), land masking from the surface
`hFacC`, and the tile edge rim.

**Tile edge rim:** every field here is purely vertical — no horizontal
stencil — so `field_registry` gives them `edge_margin = 0` and nothing
is lost at the tile boundary.  That changes from `frontal_structure`
onwards, where the gradient chains carry a 3-cell rim.

**Gradient artifacts:** none of these fields takes a *horizontal*
gradient, so the sparkle cases in `docs/Gradients.md` do not arise
here.  They start at `frontal_structure` and `kinematic`.

**Depth clipping:** the store keeps only the top 51 levels (≈969 m).
Any mixed layer deeper than that is clipped — visible in winter
deep-convection regions, not in the Gulf Stream in November.


## Section 5 — Per-field validation

Three figures per field.

**Figure 1 — maps.**  Columns are the dependency chain, raw → final.
Rows are the four depth levels over the whole tile, then the same four
zoomed to a 200 × 200 km box; the crimson square on the whole-tile rows
is where the zoom is.  One colour scale per column, shared by every row
including the zooms, so nothing changes colour when you look closer.

Fields that **do not vary with depth** get two rows instead of eight —
whole tile and zoom.  `mixed_layer_depth` and `ml_heat_content` are
both integrals over the entire water column, so four identical depth
rows would say nothing.  Their 3D chain inputs are shown at the surface
in those figures, and the title says so.

**Figure 2 — PDFs.**  Same columns; four rows, the whole tile at each
level.  Bins are shared down a column, so reading a column top to
bottom shows how the distribution changes with depth.  The zoom boxes
are deliberately absent — too few cells to make an honest histogram.

**Figure 3 — profiles.**  Five ocean columns, spread across the tile
and fixed by a seed so every field profiles the same water.  The
leftmost panel shows where they are, as numbered colour-coded ×; then
one panel per 3D field in the chain, with the **surface at the top and
depth increasing downward**.  Dashed horizontal lines mark each
location's mixed-layer depth.  The location numbers repeat in the
legend, so the five are distinguishable without relying on colour.

("Grid" in the function names below means the rows × columns array of
panels — not the model's Arakawa C-grid, which is `docs/Grid.md`.)


In [ ]:
# Section 5 helpers: one call per figure, shared by every field.
CHAINS = {
    "N2": ["Theta", "Salt", "rho", "drho_dz", "N2"],
    "mixed_layer_depth": ["Theta", "Salt", "rho", "mixed_layer_depth"],
    "ml_heat_content": ["Theta", "mixed_layer_depth", "ml_heat_content"],
}
LOG_FIELDS = set()

# Fields with no depth dependence -- integrals over the whole column.
# Their figures collapse to 2 rows (whole tile + zoom) / 1 PDF row.
DEPTH_INVARIANT = {"mixed_layer_depth", "ml_heat_content"}


def figure1_maps(field):
    """Figure 1: chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    note = (" | depth-invariant: 3D inputs shown at the surface"
            if flat else "")
    dfig.depth_map_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        region=REGION,
        levels=("sfc",) if flat else LEVELS,
        row_labels=(("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom")
                    if flat else None),
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        zoom_half_km=ZOOM_HALF_KM,
        suptitle=(f"Figure 1 — {field} | {REGION} tile | columns = "
                  f"dependency chain, rows = depth{note}"),
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: PDFs, chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    dfig.depth_pdf_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        levels=("sfc",) if flat else LEVELS,
        row_labels=("whole tile",) if flat else None,
        log10_fields=LOG_FIELDS,
        suptitle=(f"Figure 2 — {field} | {REGION} tile | density; "
                  f"land + rim NaNs dropped; bins shared down each column"),
    )
    plt.show()


def figure3_profiles(field):
    """Figure 3: depth profiles at the five fixed locations."""
    dfig.depth_profile_grid(
        CHAINS[field], PROFILES, DEPTH_M, CMAP_CFG,
        points=POINTS, level_arrays=level_arrays, region=REGION,
        mld_at_points=MLD_AT_POINTS,
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        max_depth=PROFILE_MAX_DEPTH,
        suptitle=(f"Figure 3 — {field} | {REGION} tile | profiles at "
                  f"{len(POINTS)} locations; surface at top"),
    )
    plt.show()

In [ ]:
# Safety net: every field named in a chain must actually have been
# computed, or the figure call fails deep inside matplotlib.
_missing = sorted({f for c in CHAINS.values() for f in c}
                  - set(level_arrays))
assert not _missing, f"chain fields never computed: {_missing}"
print(f"chains OK : {len(CHAINS)} fields, "
      f"{len({f for c in CHAINS.values() for f in c})} distinct columns")

### N² — buoyancy frequency squared

**N² = (g/ρ₀)·∂ρ/∂z**  [s⁻²] — positive-downward z, so no leading minus (see Section 4).

Stratification strength.  Expect the `at MLD` row to be the *strongest* of the four: the mixed-layer depth is by definition where density starts changing sharply, so that level sits in the pycnocline.  The `surface` and `MLD mean` rows sample well-mixed water and should be much weaker.  Negative values are real (statically unstable cells) and should be sparse and speckled, not organised.  In the profiles, look for the near-zero segment above the dashed MLD line and the peak at it.

In [ ]:
figure1_maps("N2")

In [ ]:
figure2_pdfs("N2")

In [ ]:
figure3_profiles("N2")

### MLD — mixed layer depth

**MLD = max{ z : σ₀(z) − σ₀(10 m) ≤ 0.03 kg m⁻³ }**  [m]

The field every other depth notebook leans on, so it is validated here and referenced elsewhere.  On the Gulf Stream tile expect deeper values north of the front and shallow values in the warm core; sharp lateral steps across the front are physical.  Two rows only — MLD is an integral over the whole column and has no depth dependence.

In [ ]:
figure1_maps("mixed_layer_depth")

In [ ]:
figure2_pdfs("mixed_layer_depth")

In [ ]:
figure3_profiles("mixed_layer_depth")

### Q_ml — mixed-layer heat content

**Q_ml = ∫₀^MLD c_p·ρ₀·Θ dz**  [J m⁻²]

Inherits both the MLD pattern and the temperature pattern: large where the layer is both deep and warm.  Also depth-independent, so two rows.  Watch for exactly-zero cells — see the note under the checks below; they are a defect in the field, not a physical result.

In [ ]:
figure1_maps("ml_heat_content")

In [ ]:
figure2_pdfs("ml_heat_content")

In [ ]:
figure3_profiles("ml_heat_content")

## Section 5b — Vertical stencil check

The surface validation found "sparkles": interpolating a finite
difference across the same axis it was differenced along makes the two
neighbouring slopes (−a and +a at an extremum) average to nearly zero,
and squaring turns that into a speck.  `docs/Gradients.md` has the four
cases and which ones we fixed.

**The vertical has the same problem built into the stencil.**  The
production vertical derivative is centred at interior levels:

    (f[k+1] − f[k−1]) / (z[k+1] − z[k−1])

On even spacing that is *identically* the mean of the two one-sided
slopes either side of level k — the same −a/+a cancellation.  The
difference from the horizontal case is that there is no separate
interpolation step to move: the stencil **is** the interpolation, and
it never reads level k.  A one-level inversion or a sharp step is
invisible to it.

So this section measures it rather than eyeballing it, on potential density (the N² source):

| Column | What |
|---|---|
| `dz_centred` | what the pipeline computes |
| `dz_onesided` | the larger-magnitude one-sided slope straddling level k — what a non-cancelling estimate reports |
| `dz_loss` | `1 − |centred| / |one-sided|`.  **0** = the two agree, **1** = the centred form cancelled away entirely |

`dz_loss` is the map to read.  Expect it near zero in smooth water and
large in a thin band at the pycnocline — which is exactly where the
`_mld` extraction lands.  If that is what you see, the `_mld` row of
every N²-derived field in this notebook is sitting on the noisiest
part of the stencil, and that is a property of the discretisation, not
of the ocean.


In [ ]:
# Centred vs one-sided d/dz, reduced to the same four depth levels.
AB_LABEL = "potential density (the N² source)"
AB = dfig.vertical_stencil_ab(rho, ds_merge)
ab_arrays = dfig.pack_tile_levels(
    dfig.compute_levels(AB, ds_merge, mld=mld, levels=LEVELS),
    XC, YC, edge_margin=EDGE_MARGIN, land_mask=LAND, levels=LEVELS,
    verbose=False)

dfig.depth_map_grid(
    ["dz_centred", "dz_onesided", "dz_loss"], ab_arrays, CMAP_CFG,
    region=REGION, levels=LEVELS, diverging_cmaps=DIVERGING,
    zoom_half_km=ZOOM_HALF_KM,
    suptitle=(f"Figure 4 — vertical stencil A/B on {AB_LABEL}: "
              f"centred vs one-sided d/dz, and the cancellation loss"))
plt.show()

dfig.depth_pdf_grid(
    ["dz_loss"], ab_arrays, CMAP_CFG, levels=LEVELS,
    suptitle="Figure 5 — cancellation loss by depth level")
plt.show()

# How much of the tile is materially affected, level by level.
print(f"{'level':<10}{'loss>0.25':>11}{'loss>0.50':>11}"
      f"{'median':>10}")
print("-" * 42)
for _lev in LEVELS:
    _a = ab_arrays["dz_loss"][_lev][2]
    _f = np.isfinite(_a)
    if not _f.any():
        continue
    print(f"{_lev:<10}{100 * np.nanmean(_a > 0.25):>10.1f}%"
          f"{100 * np.nanmean(_a > 0.50):>10.1f}%"
          f"{np.nanmedian(_a):>10.3f}")

## Section 6 — Literature comparison

**PENDING — nothing to build here yet.**

The comparison figure is chosen *after* the literature figure is, not
before.  Once LH picks a paper figure and drops the PNG into
`../literature_figures/` (naming convention
`{field(s)}_{Citation}_{description}.png`), we decide which of our
panels belongs beside it and add a subsection here — one subsection per
reference, using `dbof.plotting.literature_comparison.side_by_side`.

Leave this section as-is until then.


## Summary — did every channel come out sane?

Coverage and range for each channel at each level, then the physical
checks that are worth failing loudly on.


In [ ]:
# Coverage + range per field per level.
print(f"{'field':<22}{'level':<10}{'finite %':>9}"
      f"{'min':>14}{'max':>14}")
print("-" * 69)
for field in sorted(level_arrays):
    for lev in LEVELS:
        arr = level_arrays[field][lev][2]
        finite = np.isfinite(arr)
        pct = 100.0 * finite.mean()
        lo = np.nanmin(arr) if finite.any() else np.nan
        hi = np.nanmax(arr) if finite.any() else np.nan
        print(f"{field:<22}{lev:<10}{pct:>8.1f}%{lo:>14.4g}{hi:>14.4g}")

In [ ]:
# Physical checks.  These assert -- a red cell here is a real problem.
mld_arr = level_arrays["mixed_layer_depth"]["sfc"][2]
n2_sfc = level_arrays["N2"]["sfc"][2]
n2_mld = level_arrays["N2"]["mld"][2]
n2_mlm = level_arrays["N2"]["mld_mean"][2]
drho = level_arrays["drho_dz"]["z25m"][2]
q_ml = level_arrays["ml_heat_content"]["sfc"][2]

# ml_heat_content returns EXACTLY 0.0 -- not NaN -- for any column where
# the MLD mask selects no model level, because
# `(integrand * dz).where(mask).sum(dim=zdim)` uses xarray's default
# skipna=True and the sum of nothing is 0.  Treat those as missing and
# count them; see the note printed below.
q_zero = np.isfinite(q_ml) & (q_ml == 0.0)
q_real = np.where(q_zero, np.nan, q_ml)

CHECKS = [
    ("MLD strictly positive",
     np.nanmin(mld_arr) > 0,
     f"min = {np.nanmin(mld_arr):.1f} m"),
    ("MLD inside the stored water column (<= 969 m)",
     np.nanmax(mld_arr) <= 969.0,
     f"max = {np.nanmax(mld_arr):.1f} m"),
    ("drho/dz mostly positive -- confirms positive-DOWNWARD z",
     np.nanmean(drho > 0) > 0.9,
     f"{100 * np.nanmean(drho > 0):.1f}% positive at 25 m"),
    ("N2 mostly positive (statically stable) at 25 m",
     np.nanmean(level_arrays['N2']['z25m'][2] > 0) > 0.9,
     f"{100 * np.nanmean(level_arrays['N2']['z25m'][2] > 0):.1f}%"),
    ("N2 strongest AT the MLD -- the pycnocline, not the mixed layer",
     np.nanmedian(n2_mld) > np.nanmedian(n2_sfc),
     f"median N2: mld = {np.nanmedian(n2_mld):.2e}, "
     f"sfc = {np.nanmedian(n2_sfc):.2e}"),
    ("N2 at the MLD exceeds the mixed-layer mean",
     np.nanmedian(n2_mld) > np.nanmedian(n2_mlm),
     f"median N2: mld = {np.nanmedian(n2_mld):.2e}, "
     f"mld_mean = {np.nanmedian(n2_mlm):.2e}"),
    ("ML heat content positive wherever it is not a degenerate zero",
     np.nanmin(q_real) > 0,
     f"min = {np.nanmin(q_real):.3e} J m-2"),
    ("tile is not mostly land",
     np.isfinite(mld_arr).mean() > 0.5,
     f"{100 * np.isfinite(mld_arr).mean():.1f}% finite"),
]

n_zero = int(q_zero.sum())
if n_zero:
    print(f"NOTE  ml_heat_content is exactly 0.0 in {n_zero} cells "
          f"({100 * n_zero / q_zero.size:.3f}% of the tile) -- columns "
          f"where the MLD mask caught no model level.  These should be "
          f"NaN, not 0.  Excluded from the check above.\n")

failures = []
for name, ok, detail in CHECKS:
    print(f"{'OK  ' if ok else 'FAIL'}  {name}  ({detail})")
    if not ok:
        failures.append(name)
assert not failures, f"physical checks failed: {failures}"
print("\nAll physical checks passed.")

---

### Cross-references

- **MLD** is validated here and nowhere else.  Every notebook with a
  `_mld` / `_mld_mean` channel, plus `Fr` and `KE`, points back to this
  section rather than re-deriving it.
- **σ₀ / buoyancy** as output channels live in
  `surface_fields/frontal_structure.ipynb`.
- **The tile workflow** — what a tile is, how the rect index maps to a
  face, the `edge_margin` convention — `docs/Tiles.md` and
  `src/dbof/tiles/`.
- **Gradient / interpolation artifacts** — `docs/Gradients.md`; the
  evidence notebook is `../field_validation_sparkle.ipynb`.  Nothing in
  this subset takes a horizontal gradient, so none of it applies here.
- **The model grid** (C-grid staggering, the 51 stored levels and their
  real depths) — `../Grid.ipynb` and `docs/Grid.md`.
